# Curso: Introducción a Machine Learning

Jose Alejandro Alfaro Arias  
### Clase 10 - Parte I/II  
## k-Vecinos Más Cercanos: k-Nearest Neighbors (kNN)

Este notebook fue reconstruido a partir del HTML de la clase y documentado para poder estudiarlo y ejecutarlo en VS Code.

La idea es conservar la secuencia del material original y agregar explicaciones que ayuden a entender **qué hace cada bloque, por qué se hace y qué debemos observar**.

> **Nota:** cuando aparece una corrección técnica respecto al HTML original, queda indicada explícitamente en el notebook.

## 1. ¿Qué es kNN?

El método **k-Nearest Neighbors (kNN)** puede utilizarse tanto en problemas de **clasificación** como de **regresión**.

La idea básica es sencilla: para predecir un nuevo dato, el algoritmo busca los `k` ejemplos del conjunto de entrenamiento que se encuentran más cerca de ese punto.

- En **clasificación**, la predicción se decide normalmente por la clase más frecuente entre los vecinos.
- En **regresión**, suele utilizarse el promedio de los valores de los vecinos.

Documentación indicada en el material del profesor:

- `KNeighborsClassifier`: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html
- `KNeighborsRegressor`: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html

En esta primera parte de la clase trabajaremos con **clasificación**.

## 2. Importación de librerías

Primero cargamos las herramientas necesarias para generar los datos, visualizarlos, dividirlos en entrenamiento y prueba, construir el modelo KNN y evaluar sus resultados.

In [ ]:
# MANEJO NUMÉRICO Y VISUALIZACIÓN
import numpy as np
import matplotlib.pyplot as plt

# DATASETS Y HERRAMIENTAS DE SCIKIT-LEARN
import sklearn.datasets

# Permite separar los datos en conjuntos de entrenamiento y prueba.
from sklearn.model_selection import train_test_split

# Modelo K-Nearest Neighbors para clasificación.
from sklearn.neighbors import KNeighborsClassifier

# Herramienta para evaluar clasificación mediante una matriz de confusión.
# En esta parte del notebook se importa, aunque no llega a utilizarse después.
from sklearn.metrics import confusion_matrix

# En el HTML original también aparece la línea "#import ML".
# Al estar comentada, Python no la ejecuta y no afecta el notebook.

## 3. Creación de un conjunto de datos artificial

Para estudiar KNN, el profesor utiliza `make_moons()` de scikit-learn.

Esta función genera dos grupos de puntos con forma de **medias lunas**, lo que resulta útil porque permite probar un clasificador sobre una frontera de decisión que no es simplemente una línea recta.

El parámetro `noise` agrega ruido a los datos, haciendo que las clases se mezclen un poco y que el problema sea más realista.

In [ ]:
# Número total de observaciones que vamos a generar.
N = 1000

# Generamos un problema de clasificación binaria con forma de dos medias lunas.
noisy_data = sklearn.datasets.make_moons(
    n_samples=N,       # Cantidad total de datos.
    noise=.23,         # Nivel de ruido: aumenta la mezcla entre las dos clases.
    random_state=23    # Permite reproducir exactamente los mismos datos.
)

# make_moons devuelve dos elementos:
# X -> variables predictoras (coordenadas de cada punto).
# Y -> clase a la que pertenece cada punto: 0 o 1.
X, Y = noisy_data

# Revisamos las dimensiones de ambos objetos.
print(X.shape)
print(Y.shape)

### ¿Qué significan las dimensiones?

Como generamos `1000` observaciones y cada punto tiene dos coordenadas, esperamos que:

- `X` tenga forma `(1000, 2)`.
- `Y` tenga forma `(1000,)`.

Las dos columnas de `X` serán las variables que el modelo utilizará para determinar cuáles puntos están más cerca entre sí.

In [ ]:
# Graficamos los datos.
# X[:, 0] representa la primera variable y X[:, 1] la segunda.
# El argumento c=Y colorea cada punto según su clase.
plt.figure(figsize=(8, 5))
plt.scatter(
    X[:, 0],
    X[:, 1],
    c=Y,
    s=20,
    cmap=plt.cm.Spectral
)

plt.title("Datos generados con make_moons")
plt.xlabel("Variable X1")
plt.ylabel("Variable X2")
plt.show()

## 4. División de los datos: entrenamiento y prueba

Ahora separamos los datos en dos grupos:

- **Training set:** datos que el algoritmo utiliza para construir el modelo.
- **Test set:** datos que no se usan para entrenarlo y que permiten evaluar qué tan bien generaliza.

En el HTML original se utiliza un **60% para entrenamiento** y el restante **40% para prueba**.

> El material original no fija `random_state` en esta división. Por eso, si ejecutas el notebook varias veces, esta partición puede cambiar y los scores también pueden variar ligeramente.

In [ ]:
# Separamos el 60% de los datos para entrenamiento.
# El 40% restante se asigna automáticamente al conjunto de prueba.
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    train_size=.60
)

# Comprobamos las dimensiones de cada conjunto.
print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_test :", X_test.shape)
print("Y_test :", Y_test.shape)

## 5. Primer modelo KNN

Construimos inicialmente un KNN con:

`n_neighbors = 3`

Eso significa que, para clasificar un nuevo punto, el modelo busca sus **3 vecinos más cercanos** y utiliza la información de esos vecinos para tomar la decisión.

In [ ]:
# Creamos un clasificador KNN que utilizará los 3 vecinos más cercanos.
modelo_kNN = KNeighborsClassifier(
    n_neighbors=3
)

# Entrenamos el modelo.
# En KNN, "entrenar" significa principalmente almacenar y organizar
# los datos de entrenamiento para poder buscar vecinos posteriormente.
modelo_kNN.fit(X_train, Y_train)

# score() devuelve la exactitud (accuracy) del clasificador.
print("Score entrenamiento:", modelo_kNN.score(X_train, Y_train))
print("Score prueba       :", modelo_kNN.score(X_test, Y_test))

### ¿Cómo interpretar estos dos scores?

Aquí conviene comparar el desempeño en:

- **Entrenamiento**, donde el modelo ya conoce los datos.
- **Prueba**, donde debe responder ante observaciones no utilizadas en el ajuste.

Si el score de entrenamiento fuera muy alto y el de prueba considerablemente menor, podríamos sospechar **sobreajuste (overfitting)**.

El número de vecinos `k` es un **hiperparámetro**, es decir, una configuración que elegimos antes del entrenamiento y que puede modificar de forma importante el comportamiento del modelo.

En un problema de clasificación binaria suele ser conveniente probar valores impares de `k`, porque con un número par podrían aparecer empates entre las dos clases.

## 6. Probando diferentes valores de `k`

En lugar de asumir que `k=3` es la mejor opción, podemos entrenar varios modelos y comparar su desempeño.

El profesor prueba valores de `k` desde `1` hasta `15`.

In [ ]:
# Recorremos distintos valores de k.
for k in range(1, 16):

    # Para cada valor creamos un nuevo modelo KNN.
    modelo_kNN = KNeighborsClassifier(
        n_neighbors=k
    )

    # Entrenamos el modelo con el mismo conjunto de entrenamiento.
    modelo_kNN.fit(X_train, Y_train)

    # Medimos el score tanto en entrenamiento como en prueba.
    trs = modelo_kNN.score(X_train, Y_train)
    tes = modelo_kNN.score(X_test, Y_test)

    # Mostramos los resultados para comparar cada valor de k.
    print(
        'k=%d\tTrain: %.4f\tTest: %.4f'
        % (k, trs, tes)
    )

### ¿Qué estamos buscando?

No necesariamente queremos el valor de `k` que produzca el score más alto en entrenamiento.

Un `k` muy pequeño puede hacer que el modelo se adapte demasiado a casos particulares del conjunto de entrenamiento. En cambio, un `k` demasiado grande puede suavizar demasiado la decisión.

Por eso interesa encontrar un punto en el que el modelo:

1. tenga buen desempeño en datos nuevos;
2. no dependa demasiado de los datos de entrenamiento;
3. mantenga una diferencia razonable entre train y test.

En lugar de revisar manualmente todos los valores, scikit-learn ofrece herramientas de búsqueda de hiperparámetros.

# 7. Grid Search

`GridSearchCV` permite probar sistemáticamente varias combinaciones de hiperparámetros.

En este ejemplo se evaluarán combinaciones de:

- `n_neighbors`: cantidad de vecinos.
- `weights`: forma de asignar importancia a los vecinos.
- `metric`: métrica utilizada para calcular la distancia.

La validación cruzada (`cv`) permite evaluar cada combinación utilizando diferentes particiones del conjunto de entrenamiento.

In [ ]:
# Importamos GridSearchCV.
from sklearn.model_selection import GridSearchCV

In [ ]:
# Definimos la malla de hiperparámetros que queremos probar.
dicc_grid = {
    'n_neighbors': [1, 3, 5, 7, 9, 11],

    # uniform:
    # todos los vecinos tienen el mismo peso.
    #
    # distance:
    # los vecinos más cercanos reciben mayor influencia.
    'weights': ['uniform', 'distance'],

    # Dos formas distintas de medir distancia entre puntos.
    'metric': ['euclidean', 'manhattan']
}

# Creamos el modelo base.
kNN = KNeighborsClassifier()

# GridSearchCV entrenará un KNN para cada combinación del diccionario.
# cv=3 significa que se utiliza validación cruzada con 3 particiones.
modelo_kNN = GridSearchCV(
    kNN,
    param_grid=dicc_grid,
    cv=3
)

# Realizamos la búsqueda usando únicamente el conjunto de entrenamiento.
modelo_kNN.fit(X_train, Y_train)

# best_params_ contiene la combinación con mejor desempeño medio.
# best_score_ contiene el score promedio obtenido mediante cross-validation.
print(
    "Los mejores parametros son %s con un score de %0.2f"
    % (modelo_kNN.best_params_, modelo_kNN.best_score_)
)

## 8. Verificación del modelo encontrado

Encontrar la combinación que obtuvo el mejor resultado en `GridSearchCV` no significa que debamos aceptarla automáticamente.

Todavía debemos revisar si el modelo generaliza bien y si existe una diferencia importante entre entrenamiento y prueba.

En el HTML original, después de la búsqueda se crea manualmente un modelo llamado `kNN2`.

In [ ]:
# Creamos manualmente un KNN con una combinación de hiperparámetros
# seleccionada después de revisar la búsqueda.
kNN2 = KNeighborsClassifier(
    n_neighbors=5,
    metric='manhattan',
    weights='distance'
)

# Entrenamos este modelo.
kNN2.fit(X_train, Y_train)

# AJUSTE TÉCNICO RESPECTO AL HTML ORIGINAL:
# El HTML crea y entrena kNN2, pero luego imprime los scores usando
# la variable modelo_kNN, que todavía corresponde al GridSearchCV.
#
# Para evaluar realmente el modelo kNN2, usamos kNN2.score().
print("Score entrenamiento:", kNN2.score(X_train, Y_train))
print("Score prueba       :", kNN2.score(X_test, Y_test))

# 9. Randomized Grid Search

Otra posibilidad es `RandomizedSearchCV`.

La diferencia principal es que `GridSearchCV` intenta todas las combinaciones de la malla, mientras que `RandomizedSearchCV` selecciona solamente una cantidad determinada de combinaciones de forma aleatoria.

Esto puede ser útil cuando tenemos muchos hiperparámetros y probar todas las combinaciones sería costoso.

En el ejemplo del profesor se amplía la búsqueda incluyendo también el parámetro `algorithm`.

In [ ]:
# Importamos RandomizedSearchCV.
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
# Espacio de hiperparámetros que podrá explorar la búsqueda aleatoria.
dicc_grid = {
    'n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15, 17],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
}

# 9 valores de n_neighbors
# x 2 tipos de weights
# x 2 métricas
# x 4 algoritmos
# = 144 combinaciones posibles.

kNN = KNeighborsClassifier()

modelo_kNN = RandomizedSearchCV(
    kNN,

    # Conjunto de posibilidades del que se seleccionarán combinaciones.
    param_distributions=dicc_grid,

    # Cross-validation con 5 particiones.
    cv=5,

    # En lugar de probar las 144 combinaciones,
    # seleccionamos solamente 20.
    n_iter=20,

    # Se agrega random_state para que la selección aleatoria de las
    # combinaciones sea reproducible.
    # Esta línea no aparece en el HTML original; es una mejora de reproducibilidad.
    random_state=23
)

# Ejecutamos la búsqueda.
modelo_kNN.fit(X_train, Y_train)

print(
    "Los mejores parametros son %s \ncon un score de %0.2f"
    % (modelo_kNN.best_params_, modelo_kNN.best_score_)
)

## 10. Modelo final de ejemplo

Finalmente, el HTML construye otro `KNeighborsClassifier` a partir de una combinación concreta de hiperparámetros.

De nuevo, es importante evaluar **ese modelo específico** y no la variable de búsqueda utilizada anteriormente.

In [ ]:
# Modelo final mostrado en el material.
kNN2 = KNeighborsClassifier(
    n_neighbors=11,
    metric='euclidean',
    weights='distance',
    algorithm='brute'
)

# Entrenamos el modelo final.
kNN2.fit(X_train, Y_train)

# AJUSTE TÉCNICO RESPECTO AL HTML ORIGINAL:
# En el HTML se vuelve a imprimir modelo_kNN.score(), aunque se acaba
# de entrenar kNN2. Aquí usamos kNN2.score() para evaluar el modelo correcto.
print("Score entrenamiento:", kNN2.score(X_train, Y_train))
print("Score prueba       :", kNN2.score(X_test, Y_test))

# Resumen de la clase

En esta primera parte sobre KNN se trabajan varias ideas importantes de Machine Learning:

1. **KNN clasifica según la cercanía entre observaciones.**
2. El número de vecinos `k` es un **hiperparámetro**.
3. Elegir un valor de `k` demasiado pequeño o demasiado grande puede afectar la capacidad de generalización.
4. No debemos evaluar un modelo únicamente con los datos de entrenamiento.
5. `GridSearchCV` permite probar sistemáticamente combinaciones de hiperparámetros.
6. `RandomizedSearchCV` permite explorar solo una parte de las combinaciones cuando el espacio de búsqueda es grande.
7. La **validación cruzada** ayuda a comparar configuraciones de manera más robusta.
8. Una vez elegido un modelo, siempre debemos verificar su desempeño sobre datos que no hayan sido utilizados directamente para entrenarlo.

## Parámetros de KNN que aparecen en la clase

- `n_neighbors`: cantidad de vecinos considerados.
- `weights='uniform'`: todos los vecinos pesan igual.
- `weights='distance'`: los vecinos más cercanos tienen mayor influencia.
- `metric='euclidean'`: utiliza distancia euclidiana.
- `metric='manhattan'`: utiliza distancia Manhattan.
- `algorithm`: estrategia interna que scikit-learn utiliza para buscar vecinos.

La idea central no es memorizar una combinación específica de parámetros, sino entender que **los hiperparámetros deben evaluarse y compararse antes de decidir cuál modelo utilizar**.